**Phase 3e: Index Entities in Azure AI Search**
- Synapse Notebook: 03_e_gold_index_entities
- Azure AI Search kg-entities index creation + upload.
- Trigger: After 03_d_gold_index_chunks completes
- Input:   gold_llm/knowledge_graph/nodes.json on ADLS
- Output:  Azure AI Search index "kg-entities"


**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
print("Environment variables set.")

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Download nodes.json from ADLS**

In [ ]:
GOLD_PATH = "gold_llm"
LOCAL_GOLD = "/tmp/pipeline/gold"

os.makedirs(f"{LOCAL_GOLD}/knowledge_graph", exist_ok=True)

# Download nodes.json
nodes_data = adapter.read_bytes(f"{GOLD_PATH}/knowledge_graph/nodes.json")
with open(f"{LOCAL_GOLD}/knowledge_graph/nodes.json", "wb") as fh:
    fh.write(nodes_data)

print(f"Downloaded nodes.json from ADLS")

**Create index + embed + upload entities**

In [ ]:
import json
from pathlib import Path
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)
from azure.search.documents import SearchClient
from openai import AzureOpenAI

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
api_key = os.environ["AZURE_SEARCH_API_KEY"]
credential = AzureKeyCredential(api_key)
entity_index_name = os.environ.get("AZURE_SEARCH_ENTITY_INDEX_NAME", "kg-entities")

index_client = SearchIndexClient(endpoint, credential)

try:
    index_client.delete_index(entity_index_name)
    print(f"Deleted existing index '{entity_index_name}'")
except Exception:
    pass

fields = [
    SearchField(name="entity_id", type=SearchFieldDataType.String, key=True, filterable=True, searchable=True),
    SearchField(name="name", type=SearchFieldDataType.String, searchable=True, filterable=True),
    SearchField(name="node_type", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name="mention_count", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SearchField(name="entity_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),searchable=True, retrievable=True, stored=True, vector_search_dimensions=1536, vector_search_profile_name="hnsw-profile"),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-algo")],
    profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-algo")],
)

index = SearchIndex(name=entity_index_name, fields=fields, vector_search=vector_search)
index_client.create_or_update_index(index)
print(f"Entity index '{entity_index_name}' created")

# Embedding client
oai_client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-12-01-preview",
)
emb_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
EMBED_BATCH_SIZE = 16

def embed_batch(texts):
    truncated = [t[:8000] for t in texts]
    response = oai_client.embeddings.create(input=truncated, model=emb_deployment)
    return [d.embedding for d in response.data]

# Load node metadata
with open(f"{LOCAL_GOLD}/knowledge_graph/nodes.json") as f:
    nodes = json.load(f)

# Collect entities (skip internal types)
search_client = SearchClient(endpoint, entity_index_name, credential)
entities_to_process = []
skipped = 0

for node_id, node in nodes.items():
    node_type = node.get("node_type", "UNKNOWN")
    if node_type in ("CHUNK", "THREAD", "EMAIL"):
        skipped += 1
        continue

    name = node.get("name", node_id)
    entities_to_process.append({
        "entity_id": node_id,
        "name": name,
        "node_type": node_type,
        "mention_count": node.get("mention_count", 0),
        "embed_input": f"{node_type}: {name}",
    })

print(f"Loaded {len(entities_to_process)} entities ({skipped} internal skipped), embedding in batches of{EMBED_BATCH_SIZE}...")

# Batch embed + stream upload
documents = []
total_ok = 0

for i in range(0, len(entities_to_process), EMBED_BATCH_SIZE):
    batch = entities_to_process[i:i + EMBED_BATCH_SIZE]

    vectors = embed_batch([r["embed_input"] for r in batch])

    for j, r in enumerate(batch):
        documents.append({
            "entity_id": r["entity_id"],
            "name": r["name"],
            "node_type": r["node_type"],
            "mention_count": r["mention_count"],
            "entity_vector": vectors[j],
        })

    while len(documents) >= 100:
        result = search_client.upload_documents(documents[:100])
        total_ok += sum(1 for r in result if r.succeeded)
        documents = documents[100:]

    if (i // EMBED_BATCH_SIZE) % 100 == 0:
        print(f"  Progress: {min(i + EMBED_BATCH_SIZE, len(entities_to_process))}/{len(entities_to_process)} embedded,{total_ok} indexed")

if documents:
    result = search_client.upload_documents(documents)
    total_ok += sum(1 for r in result if r.succeeded)

print(f"Done — {total_ok}/{len(entities_to_process)} entities indexed")